# Week 5 Problem Set

## Homework

In [182]:
%load_ext nb_mypy
%nb_mypy Off

The nb_mypy extension is already loaded. To reload it, use:
  %reload_ext nb_mypy


In [183]:
from typing import TypeAlias
from typing import Optional, Any, Iterator
from __future__ import annotations

Number: TypeAlias = int | float
NumberList: TypeAlias = list[int|float] 

**HW1.** Modify the class `TurtleWorld` to include the following attribute and methods:
- `movement_queue` which is an attribute of the type `Queue` to store the movement list. **Each entry of this queue is an object which has the name of the turtle (e.g. "t1") and the movement list (e.g. "ulrr"). The choice of this object is left to you, e.g. it could be a tuple (`("t1", "ulrr")`) or a dictionary with the same information or your own custom class.
- `add_movement(turtle, movement)` which adds turtle movement to the queue `movement_queue` to be run later. The argument `turtle` is a string containing the turtle's name. The argument `movement` is another string for the movement. For example, value for `turtle` can be something like `'t1'` while the value for the `movement` can be something like `'uullrrdd'`.
- `run()` which executes all the movements in the queue.

In [184]:
import math

class Coordinate:
    
    def __init__(self, x:Number=0, y:Number=0) -> None:
        self.x = x
        self.y = y
        
    @property
    def distance(self) -> float:
        return math.sqrt(self.x * self.x + self.y * self.y)
    
    def __str__(self) -> str:
        return f"({self.x}, {self.y})"

In [185]:
class Queue:
    def __init__(self) -> None:
        self.__items: list[Any] = []
    
    def enqueue(self, item: Any) -> None:
        self.__items.append(item)

    def dequeue(self) -> Any:
        return self.__items.pop(0)
    
    def peek(self) -> Any:
        if not self.is_empty:
            return self.__items[0]


    @property
    def is_empty(self) -> bool:
        return self.size == 0
    
    @property
    def size(self) -> int:
        return len(self.__items)


In [186]:
# Class definition
class RobotTurtle:
    # Attributes:
    def __init__(self, name: str, speed: int=1) -> None:
        assert isinstance(name, str) and name != ""
        assert isinstance(speed, int) and speed > 0
        self._name: str = name
        self._speed: int = speed
        self._pos: Coordinate = Coordinate(0, 0)
        
    # property getter
    @property
    def name(self) -> str:
        return self._name
    
    # property setter
    @name.setter
    def name(self, value: str) -> None:
        if isinstance(value, str) and value != "":
            self._name = value
            
    # property getter
    @property
    def speed(self) -> int:
        return self._speed
    
    # property setter
    @speed.setter
    def speed(self, value: int) -> None:
        if isinstance(value, int) and value > 0:
            self._speed = value

    # property getter
    @property
    def pos(self) -> Coordinate:
        return self._pos
    
    # Methods:
    def move(self, direction: str) -> None:
        update: dict[str, Coordinate] = {'up' : Coordinate(self.pos.x, self.pos.y + self.speed),
                                        'down' : Coordinate(self.pos.x, self.pos.y - self.speed),
                                        'left' : Coordinate(self.pos.x - self.speed, self.pos.y),
                                        'right' : Coordinate(self.pos.x + self.speed, self.pos.y)}
        self._pos = update[direction]

        
    def tell_name(self) -> None:
        print(f"My name is {self.name}")


In [187]:
class TurtleWorld:
    valid_movements:set[str] = set('udlr')
    movement_map: dict[str, str] = {'u': 'up', 'd': 'down', 'l': 'left', 'r': 'right'}
    
    def __init__(self) -> None:
        self.turtles: dict[str, RobotTurtle] = {}
        self.movement_queue = Queue()
        
    def add_movement(self, turtle: str, movement: str) -> None:
        self.movement_queue.enqueue((turtle,movement))
    
    def run(self) -> None:
        while not self.movement_queue.is_empty:
            name, movement = self.movement_queue.dequeue()
            self.move_turtle(name,movement)
        pass
        
    def move_turtle(self, name: str, movement: str) -> None:
        for c in movement:
            if c in self.valid_movements:
                self.turtles[name].move(self.movement_map[c])
    
    def add_turtle(self, name: str, speed: int) -> None:
        turtle = RobotTurtle(name,speed)
        self.turtles[name] = turtle
        
    def remove_turtle(self, name: str) -> None:
        self.turtles.pop(name)
        pass
        
    def list_turtles(self) -> list[str]:
        return sorted(self.turtles.keys())
        pass

In [188]:
world: TurtleWorld = TurtleWorld()
assert isinstance(world.movement_queue, Queue)

world.add_turtle('t1', 1)
world.add_turtle('t2', 2)
world.add_movement('t1', 'ur')
world.add_movement('t2', 'urz')
assert str(world.turtles['t1'].pos) == '(0, 0)'
assert str(world.turtles['t2'].pos) == '(0, 0)'
assert world.movement_queue.size == 2

world.run()
assert str(world.turtles['t1'].pos) == '(1, 1)'
assert str(world.turtles['t2'].pos) == '(2, 2)'

world.add_movement('t1', 'ur')
world.add_movement('t2', 'urz')

world.run()
assert str(world.turtles['t1'].pos) == '(2, 2)'
assert str(world.turtles['t2'].pos) == '(4, 4)'



In [189]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**HW2.** Implement a radix sorting machine. A radix sort for base 10 integers is a *mechanical* sorting technique that utilizes a collection of bins:
- one main bin 
- 10 digit-bins

Each bin acts like a *queue* and maintains its values in the order that they arrive. The algorithm works as follows:
- it begins by placing each number in the main bin. 
- Then it considers each value digit by digit. The first value is removed from the main bin and placed in a digit-bin corresponding to the digit being considered. For example, if the ones digit is being considered, 534 will be placed into digit-bin 4 and 667 will placed into digit-bin 7. 
- Once all the values are placed into their corresponding digit-bins, the values are collected from bin 0 to bin 9 and placed back in the main bin (in that order). 
- The process continues with the tens digit, the hundreds, and so on. 
- After the last digit is processed, the main bin will contain the values in ascending order.

Create a class `RadixSort` that takes in a List of Integers during object instantiation. The class should have the following properties:
- `items`: is a List of Integers containing the numbers.

It should also have the following methods:
- `sort()`: which returns the sorted numbers from `items` as an `list` of Integers.
- `max_digit()`: which returns the maximum number of digits of all the numbers in `items`. For example, if the numbers are 101, 3, 1041, this method returns 4 as the result since the maximum digit is four from 1041. 
- `convert_to_str(items)`: which returns items as a list of Strings (instead of Integers). This function should pad the higher digits with 0 when converting an Integer to a String. For example if the maximum digit is 4, the following items are converted as follows. From `[101, 3, 1041]` to `["0101", "0003", "1041"]`.

Hint: Your implementation should make use of the generic `Queue` class, which you created, for the bins.

In [190]:
class RadixSort:
    
    def __init__(self, my_list: list[int]) -> None:
        self.items = my_list
    
    def max_digit(self) -> int:
        digits = 0 
        for i in self.items:
            digits = max(len(str(i)),digits)
        return int(digits)
    
    def convert_to_str(self, items: list[int]) -> list[str]:
        length = self.max_digit()
        return [str(n).zfill(length) for n in items]
    
    def sort(self) -> list[int]:
        temp_list = self.convert_to_str(self.items)
        for str_idx in range(self.max_digit()-1,-1,-1):
            buckets = [[] for _ in range(0,10)]
            for list_item in temp_list:
                buckets[int(list_item[str_idx])].append(list_item)
            
            temp_list = []
            for bucket in buckets:
                temp_list.extend(bucket)
        return [int(num) for num in temp_list]

In [191]:
list1: RadixSort = RadixSort([101, 3, 1041])
assert list1.items == [101,3,1041]
assert list1.max_digit() == 4
assert list1.convert_to_str(list1.items) == ["0101", "0003", "1041"]
ans: list[int] = list1.sort()
print(ans)
assert ans == [3, 101, 1041]
list2: RadixSort = RadixSort([23, 1038, 8, 423, 10, 39, 3901])
assert list2.sort() == [8, 10, 23, 39, 423, 1038, 3901]

[3, 101, 1041]


In [192]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**HW3**. Create a class `Rectangle` which is a subclass of `GeometricObject`. It has two additional properties:
- `width` with default value of 1.0.
- `height` with default value of 1.0

It also has two computed properties:
- `area`
- `perimeter`

It has one method:
- `print()` that displays the information about the rectangle in the following format: "colour: green and filled: True and area: xx.xx". Use string formating to format the output area to two decimal places. 

In [193]:
class GeometricObject:
    VALID_COLOURS: set[str] = {"red", "green", "blue", "white", "black"}

    def __init__(self, colour: str = "green", filled: bool = True) -> None:
        assert isinstance(colour, str)
        assert isinstance(filled, bool)
        self._colour = colour
        self._string = filled

    @property 
    def colour(self):
        return self._colour

    @colour.setter
    def colour(self, new_colour):
        if isinstance(new_colour,str) and new_colour in self.VALID_COLOURS:
            self._colour = new_colour
        else:
            self._colour = 'green'

    @property
    def filled(self):
        return self._string 

    @filled.setter
    def filled(self, new_bool):
        if isinstance(new_bool, bool):
            self._string = new_bool

    def __str__(self) -> str:
        return f"colour: {self.colour:s} and filled: {str(self.filled):s}"

In [262]:
class Rectangle(GeometricObject):

    def __init__(self, width: float = 1.0, height: float = 1.0,
                 colour: str = "green", filled: bool = True):
        super().__init__(colour, filled)
        self._width = float(width)
        self._height = float(height)

    @property
    def width(self) -> float:
        return self._width

    @width.setter
    def width(self, val: float) -> None:
        if isinstance(val, (int, float)):
            self._width = float(val)

    @property
    def height(self) -> float:
        return self._height

    @height.setter
    def height(self, val: float) -> None:
        if isinstance(val, (int, float)):
            self._height = float(val)

    @property
    def area(self) -> float:
        return self.width * self.height

    @property
    def perimeter(self) -> float:
        return 2 * (self.width + self.height)

    def __str__(self) -> str:
        return (f"colour: {self.colour} and "
                f"filled: {self.filled} and "
                f"area: {self.area:.2f}")

In [ ]:
r: Rectangle = Rectangle()
assert r.area == 1.0
r.width = 2.0
r.height = 3.0
assert r.area == 6.0
rect: Rectangle = Rectangle(3.0, 2.0)
rect.colour = "red"
rect.filled = True
assert rect.width == 3.0
assert rect.height == 2.0
assert rect.area == 6.0
assert rect.colour == "red"
assert rect.filled == True


In [196]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**HW4.** Write a class called `EvaluateFraction` that evaluates postfix notation implemented using Stack and Queue data structures. Postfix notation is a way of writing expressions without using parenthesis. For example, the expression `(1+2)*3` would be written as `1 2 + 3 *`. The class `EvaluateFraction` has the following method:
- `input(inp)`: which pushes the input input one at a time. For example, to create a postfix notation `1 2 + 3 *`, we can call this method repetitively, e.g. `e.input('1'); e.input('2'); e.input('+'); e.input('3'); e.input('*')`. Notice that the input is of String data type. 
- `evaluate()`: which returns the output of the expression.
- `get_fraction(inp)`: which takes in an input string and returns a `Fraction` object. 

Postfix notation is evaluated using a Stack. The input streams from `input()` are stored in a Queue. If the output of the Queue is a number, the item is pushed into the stack. If it is an operator, we will apply the operator to the two top most item n the stacks and push the result back into the stack. 

In [197]:
class Stack:
    def __init__(self) -> None:
        self.__items: list[Any] = []
        
    def push(self, item: Any):
        self.__items.append(item)

    def pop(self) -> Any:
        if not self.is_empty:
            return self.__items.pop()

    def peek(self) -> Any:
        if not self.is_empty:
            return self.__items[-1]

    @property
    def is_empty(self) -> bool:
        return self.size == 0

    @property
    def size(self):
        return len(self.__items)


In [198]:
class Queue:

    def __init__(self):
        self.in_stack = Stack()
        self.out_stack = Stack()

    def enqueue(self, item):
        self.in_stack.push(item)

    def dequeue(self):
        if self.out_stack.is_empty:
            while not self.in_stack.is_empty:
                self.out_stack.push(self.in_stack.pop())

        return self.out_stack.pop()

    def peek(self):
        if self.out_stack.is_empty:
            while not self.in_stack.is_empty:
                self.out_stack.push(self.in_stack.pop())

        return self.out_stack.peek()

    @property
    def is_empty(self):
        return self.in_stack.is_empty and self.out_stack.is_empty

    @property
    def size(self):
        return self.in_stack.size + self.out_stack.size

In [199]:
def gcd(a: int, b: int) -> int:
    if b == 0:
        return a
    else:
        return gcd(b, a % b)


class Fraction:
    def __init__(self, num: int, den: int) -> None:
        assert isinstance(num, int)
        assert isinstance(den, int)
        self._num = num
        self._den = 0 if den == 0 else den
        pass
        

    @property
    def num(self) -> int:
        return self._num
    
    @num.setter
    def num(self, val: int) -> None:
        if isinstance(val,int):
            self._num = val
    
    @property
    def den(self) -> int:
        return self._den
    
    @den.setter
    def den(self, val: int) -> None:
        if isinstance(val, (float,int)):
            if val == 0:
                self._den = 1
            else: 
                self._den = int(val)
    
    def __str__(self) -> str:
        return f'{self._num}/{self._den}'
    
    def simplify(self) -> Fraction:
        div_by = gcd(self._num,self._den)
        simple_num = self.num//div_by
        simple_den = self.den//div_by
        return Fraction(simple_num,simple_den)
    
    def __add__(self, other) -> Fraction:
        numerator = self._num * other._den + other._num * self._den
        denominator = self._den * other._den
        return Fraction(numerator, denominator).simplify()
        
    def __eq__(self, other) -> bool:
        simple_self = self.simplify()
        simple_other = other.simplify()
        return simple_self._den == simple_other._den and simple_self._num == simple_other.num
        
    def __sub__(self, other) -> Fraction:
        numerator = self._num * other._den - other._num * self._den
        denominator = self._den * other._den
        return Fraction(numerator, denominator).simplify()
    
    def __mul__(self, other) -> Fraction:
        numerator = self._num * other._num
        denominator = self._den * other._den
        return Fraction(numerator, denominator).simplify()
    
    def __lt__(self, other) -> bool:
        return self._num * other._den < other._num * self._den
    
    def __le__(self, other) -> bool:
        return self._num * other._den <= other._num * self._den

    
    def __gt__(self, other) -> bool:
        return self._num * other._den > other._num * self._den

    
    def __ge__(self, other) -> bool:
        return self._num * other._dxen >= other._num * self._den

In [ ]:
class EvaluateFraction:

    operands: str = "0123456789"
    operators: str = "+-*/"
    
    def __init__(self) -> None:
        self.expression: list[str] = []
        self.stack: Stack = Stack()

    def input(self, item: str) -> None:
        if len(item) != 1:
            item = self.get_fraction(item)
        self.expression.append(item)

    def evaluate(self) -> Number:
        while self.expression:
            pending = self.expression.pop(0)

            if isinstance(pending,Fraction):
                self.stack.push(pending)
            else:
                a = self.stack.pop()
                b = self.stack.pop()

                new = self.process_operator(a,b,pending)
                self.stack.push(new)
        return self.stack.pop()
    
    def get_fraction(self, inp: str) -> Fraction:
        num,den = inp.split('/')
        return Fraction(int(num),int(den))
    
    def process_operator(self, op1: Fraction, op2: Fraction, op: str) -> Fraction:
        result = None
        if op == '+':
            result = op2 + op1

        elif op =='-':
            result = op2 - op1
        elif op =='*':
            result = op2 * op1
        elif op == '/':
            div_fraction = Fraction(op1.den, op1.num)
            result = op2 * div_fraction

        return result

In [201]:
pe: EvaluateFraction = EvaluateFraction()
pe.input("1/2")
pe.input("2/3")
pe.input("+")
assert pe.evaluate()==Fraction(7, 6)

pe.input("1/2")
pe.input("2/3")
pe.input("+")
pe.input("1/6")
pe.input("-")
assert pe.evaluate()==Fraction(1, 1)

pe.input("1/2")
pe.input("2/3")
pe.input("+")
pe.input("1/6")
pe.input("-")
pe.input("3/4")
pe.input("*")
assert pe.evaluate()==Fraction(3, 4)

In [202]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**HW5.** Modify HW2 so that it can work with MixedFraction. Write a class called `EvaluateMixedFraction` as a subclass of `EvaluateFraction`. You need to override the following methods:
- `get_fraction(inp)`: This function should be able to handle string input for MixedFraction such as `1 1/2` or `3/2`. It should return a `MixedFraction` object.
- `evaluate()`: This function should return `MixedFraction` object rather than `Fraction` object. 

In [258]:
class MixedFraction(Fraction):

    def __init__(self, top: int, bot: int, whole: int = 0):
        top += whole * bot
        super().__init__(top, bot)

    def get_three_numbers(self) -> tuple[int, int, int]:
        top = self.num % self.den
        whole = self.num // self.den
        return (top, self.den, whole)

    def __str__(self) -> str:
        top, bot, whole = self.get_three_numbers()
        return f"{whole} {top}/{bot}"

In [259]:
class EvaluateMixedFraction(EvaluateFraction):

    def get_fraction(self, inp: str) -> MixedFraction:
        if " " in inp:
            whole, rest = inp.split()
            top, bot = rest.split("/")
            return MixedFraction(int(top), int(bot), int(whole))
        else:
            top, bot = inp.split("/")
            return MixedFraction(int(top), int(bot))

    def evaluate(self) -> MixedFraction:
        answer = super().evaluate()

        whole = answer.num // answer.den
        top = answer.num % answer.den
        bot = answer.den

        return MixedFraction(top, bot, whole)

In [260]:
pe: EvaluateMixedFraction = EvaluateMixedFraction()
pe.input("3/2")
pe.input("1 2/3")
pe.input("+")
out: MixedFraction = pe.evaluate() 
assert out == MixedFraction(1, 6, 3)
assert isinstance(out, MixedFraction)

pe.input("1/2")
pe.input("2/3")
pe.input("+")
pe.input("1 1/8")
pe.input("-")
assert pe.evaluate() == MixedFraction(1, 24)

pe.input("1 1/2")
pe.input("2 2/3")
pe.input("+")
pe.input("1 1/6")
pe.input("-")
pe.input("5/4")
pe.input("*")
assert pe.evaluate() == MixedFraction( 3, 4, 3)

In [261]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###